# Preprocessing


## **Mounting Google Drive in Google Colab**

* Imports the drive module to access Google Drive features.
* Mounts your Google Drive to the Colab file system.

In [2]:

from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


## **Importing Libraries for Text Processing & Data Handling**


This code imports important libraries used in Natural Language Processing (NLP) and data manipulation.

In [3]:
from numpy.random.mtrand import random_sample
import nltk
import pandas as pd
import re
from nltk.corpus import stopwords

## **Loading and Sampling Dataset using Pandas**

loads a dataset from Google Drive and selects a smaller random sample for analysis.

In [4]:

file_path = "/content/drive/MyDrive/comments.csv"
df = pd.read_csv(
    file_path,
    usecols=["body", "created_utc", "subreddit"]
)
df = df.sample(n = 200000, random_state=42)
df.reset_index(drop=True, inplace=True)
print(df.shape)
df.head()

(200000, 3)


,subreddit,body,created_utc
0,worldnews,> and overnight the narrative was accepted. N...,2025-12-15 15:10:46
1,worldnews,"U are away your taxes don’t pay for anything, ...",2025-01-21 17:28:22
2,worldnews,That’s so incredibly stupid. During ww2 Europe...,2025-03-05 03:46:40
3,worldnews,> The source said the Canadian Security Intell...,2025-03-26 00:37:25
4,worldnews,We make our own,2025-03-08 08:39:05


SAMPLED DATA SAVED

In [5]:
df.to_csv("/content/drive/MyDrive/sample_200k.csv",
    index=False)
print("Done")

Done


## **Datetime Conversion and Feature Extraction**

1. Convert raw timestamps into meaningful date format
2. Create new features (year, month, day) for analysis

In [6]:
df["date"] = pd.to_datetime(
    df["created_utc"]
)
print("Start:", df["date"].min())
print("End:", df["date"].max())
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
print(df.columns)
df['created_utc'].head(5)

Start: 2025-01-01 03:19:05
End: 2026-02-28 23:26:53
Index(['subreddit', 'body', 'created_utc', 'date', 'year', 'month', 'day'], dtype='object')


,created_utc
0,2025-12-15 15:10:46
1,2025-01-21 17:28:22
2,2025-03-05 03:46:40
3,2025-03-26 00:37:25
4,2025-03-08 08:39:05


## **Text Preprocessing and Cleaning using NLTK**

1. Clean raw text data
2. Remove noise (URLs, symbols, common words)
3. Prepare data for NLP tasks (sentiment analysis, ML models)

In [7]:
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

# Start fresh from original column
df['clean_text'] = df['body'].apply(lambda x: str(x).lower())                            # Step 1: lowercase
df['clean_text'] = df['clean_text'].apply(lambda x: re.sub(r'http\S+|www\S+', '', x))   # Step 2: remove URLs
df['clean_text'] = df['clean_text'].apply(lambda x: re.sub(r'[^a-z\s]', '', x))         # Step 3: remove special chars
df['clean_text'] = df['clean_text'].apply(lambda x: ' '.join([w for w in x.split() if w not in stop_words]))  # Step 4: stopwords

print(df['clean_text'].head(3))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


0    overnight narrative accepted wasnt hardly anyo...
1    u away taxes dont pay anything government make...
2    thats incredibly stupid ww europe divided diff...
Name: clean_text, dtype: object


## **Removing Empty and Invalid Text Rows**

1. Ensure dataset contains only valid text data
2. Remove empty or useless records
3. Improve quality of data for analysis

In [8]:
df = df[df['clean_text'].notna()]
df = df[df['clean_text'].str.strip() != '']
print("Rows after cleaning:", df.shape)

Rows after cleaning: (199216, 8)


## **Filtering Out [removed] Entries from Dataset**

1. Remove deleted or unavailable comments
2. Clean dataset from meaningless placeholder text
3. Improve quality of NLP dataset

In [9]:
df = df[df['clean_text'] != 'removed']
print("After removing [removed] rows:", df.shape)

After removing [removed] rows: (170666, 8)


## **DATE CONVERTED TO TIMESTAMP**

1. Enable time-based operations (sorting, filtering, grouping).
2. Prepare data for trend analysis over time (daily, monthly, yearly).
3. Make timestamps readable and usable in analysis.

In [10]:
df['date'] = pd.to_datetime(df['created_utc'])
df['timestamp'] = pd.to_datetime(df['created_utc'])

## **Saving Final Cleaned Dataset to CSV**

1. Store the final processed dataset permanently
2. Make data ready for future analysis or modeling
3. Avoid repeating the cleaning process

In [11]:
df.to_csv('/content/drive/MyDrive/clean_reddit_data.csv', index=False)
print("Done! Final shape:", df.shape)

Done! Final shape: (170666, 9)


# **LDA**

## **Text Vectorization Preparation for LDA Topic ModelingLDA**

1. Load clean_reddit_data.csv containing preprocessed Reddit comments.
2. Drop rows with NaN values in clean_text column.
3. Remove rows containing only whitespace or empty strings.

In [12]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Check if df exists from preprocessing
try:
    print(f"Using existing DataFrame with shape: {df.shape}")
    print(f"Columns available: {df.columns.tolist()}")

    # Check if clean_text exists
    if 'clean_text' not in df.columns:
        raise ValueError("clean_text column not found - run preprocessing first")

except NameError:
    print("DataFrame not found in memory. Loading from file...")
    # Try multiple locations
    import os
    possible_paths = [
        '/content/drive/MyDrive/clean_reddit_data.csv',
        '/content/drive/MyDrive/worlddataset/clean_reddit_data.csv',
        'clean_reddit_data.csv'
    ]

    loaded = False
    for path in possible_paths:
        if os.path.exists(path):
            df = pd.read_csv(path)
            print(f"✅ Loaded from: {path}")
            loaded = True
            break

    if not loaded:
        raise FileNotFoundError("Could not find clean_reddit_data.csv anywhere. Run preprocessing first!")

# Remove empty rows for LDA
df = df.dropna(subset=['clean_text'])
df = df[df['clean_text'].str.strip() != '']
print(f"Final shape for LDA: {df.shape}")

Using existing DataFrame with shape: (170666, 9)
Columns available: ['subreddit', 'body', 'created_utc', 'date', 'year', 'month', 'day', 'clean_text', 'timestamp']
Final shape for LDA: (170666, 9)


## **Stopword removing**

### **Text Preprocessing Pipeline for Topic Modeling**

Pipeline Stages:

1. Stopword Loading – Merges custom files (stopwords.txt, trend_stopwords.txt) + NLTK base + manual filters.
2. Critical Term Protection – Preserves geopolitical/military/econ terms (e.g., nuclear, invasion, tariff, Putin).
3. Text Cleaning – Removes URLs, punctuation, numbers, short words, negations.
4. Lemma Fixes – Corrects spaCy errors (taxes→tax, data→data).
5. Length Filtering – Retains only documents with ≥5 tokens.

In [13]:
import spacy
import re
import pandas as pd
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

nlp = spacy.load('en_core_web_sm')
nlp.max_length = 2000000

# 1. Tumhari dono files load karo
with open('/content/drive/MyDrive/stopwords.txt', 'r', encoding='utf-8', errors='ignore') as f:
    file1_words = {line.strip().lower() for line in f if line.strip()}

with open('/content/drive/MyDrive/trend_stopwords.txt', 'r', encoding='utf-8', errors='ignore') as f:
    file2_words = {line.strip().lower() for line in f if line.strip()}

# 2. Extra jo abhi bhi aa rahe the
extra_remove = {
    'not','hardly','actually','never','nothing','nobody','nowhere',
    'neither','nor','cannot','cant','wont','dont','isnt','arent',
    'wasnt','werent','havent','hasnt','hadnt','nope','yeah','yep',
    'say','said','says','saying','tell','told','ask','asked',
    'think','thought','thinking','know','knew','known',
    'conservative','normal','rational','currently','generally',
    'care','thing','things','stuff','way','lot','bit','kind',
    'human','person','people','man','men','woman','women','guy','guys',
    'taxis','datum','medium','corpus','erratum',
    'lol','lmao','omg','wtf','smh','imo','imho','reddit',
    'comment','post','edit','deleted','removed','upvote','downvote',
}

# 3. NLTK base
nltk_base = set(stopwords.words('english'))

# 4. Sab ek jagah merge
all_stopwords = nltk_base | file1_words | file2_words | extra_remove

# 5. Spacy mein inject
nlp.Defaults.stop_words.update(all_stopwords)
for word in all_stopwords:
    if word in nlp.vocab:
        nlp.vocab[word].is_stop = True

# 6. Important trend words protect karo
keep_these = {
    'nuclear','missile','invasion','sanction','tariff',
    'election','military','weapon','troop','soldier',
    'president','democracy','republican','democrat',
    'zelensky','trump','putin','nato','dictator',
    'poland','china','russia','ukraine','india','iran',
    'israel','taiwan','pakistan','japan','korea',
    'economy','trade','oil','energy','market','inflation',
    'border','territory','regime','protest','corruption',
    'court','office','national','security','power',
    'history','evidence','policy','propaganda','intelligence',
    'canadian','leadership','community','assessment','treaty',
    'alliance','occupation','ceasefire','airstrike','offensive',
    'casualties','referendum','sovereignty','annexation',
}
for word in keep_these:
    if word in nlp.vocab:
        nlp.vocab[word].is_stop = False

# 7. Spacy broken lemma fixes
lemma_fixes = {
    'taxes':'tax',  'taxis':'tax',
    'data':'data',  'datum':'data',
    'media':'media','medium':'media',
    'corps':'corps',
}

def process_batch(texts, batch_size=1000):
    results = []
    clean_texts = []

    for t in texts:
        if not isinstance(t, str):
            clean_texts.append('')
            continue
        t = t.lower()
        t = re.sub(r'http\S+|www\S+|https\S+', '', t)
        t = re.sub(r'\[.*?\]\(.*?\)', '', t)
        t = re.sub(r"n't|n't", '', t)
        t = re.sub(r"'s|'s", '', t)
        t = re.sub(r'[^a-z\s]', '', t)
        t = re.sub(r'\d{4}-\d{2}-\d{2}', '', t)
        t = re.sub(r'\b\w{1,2}\b', '', t)
        t = re.sub(r'\s+', ' ', t).strip()
        clean_texts.append(t)

    for doc in nlp.pipe(clean_texts, batch_size=batch_size,
                        disable=['ner','parser','senter']):
        tokens = []
        for token in doc:
            if token.is_stop or token.is_punct or token.is_space:
                continue
            lemma = lemma_fixes.get(token.text, token.lemma_)
            lemma = lemma_fixes.get(lemma, lemma)
            if lemma in all_stopwords:
                continue
            if len(lemma) < 3:
                continue
            tokens.append(lemma)

        results.append(" ".join(tokens) if tokens else None)

    return results

df['clean_text'] = process_batch(df['body'])
df = df[df['clean_text'].notna() & (df['clean_text'].str.strip() != '')].reset_index(drop=True)
# 1. pretty aur aise baaki fillers add karo
extra_remove = {
    'pretty','quite','rather','fairly','somewhat','slightly',
    'clearly','simply','obviously','literally','basically',
    'certainly','definitely','probably','possibly','perhaps',
    'mostly','mainly','largely','generally','typically',
    'country','countries',   # bahut generic hain trend ke liye
}

all_stopwords = all_stopwords | extra_remove

# Spacy mein update karo
for word in extra_remove:
    if word in nlp.vocab:
        nlp.vocab[word].is_stop = True

# 2. Row 4 jaisi choti rows drop karo — minimum 5 words hone chahiye
df['clean_text'] = process_batch(df['body'])

df = df[
    df['clean_text'].notna() &
    (df['clean_text'].str.strip() != '') &
    (df['clean_text'].str.split().str.len() >= 5)   # ← minimum 5 words
].reset_index(drop=True)

extra_remove = {
    # Generic verbs jo abhi bhi aa rahe hain
    'bring','brought','bringing',
    'hope','hoping','hoped',
    'option','options',
    'choose','chose','chosen',
    # Food words — worldnews trend mein useless
    'cheese','poutine','food','meal','eat','drink',
    'bread','meat','rice','fruit','vegetable',
    # Generic nouns
    'transaction','transact',
}

all_stopwords = all_stopwords | extra_remove
for word in extra_remove:
    if word in nlp.vocab:
        nlp.vocab[word].is_stop = True

# Dobara process karo
df['clean_text'] = process_batch(df['body'])

df = df[
    df['clean_text'].notna() &
    (df['clean_text'].str.strip() != '') &
    (df['clean_text'].str.split().str.len() >= 5)
].reset_index(drop=True)

print(f"✅ Done: {len(df)} rows")
print(df['clean_text'].head(10))  # 10 rows dekho ab


✅ Done: 95008 rows
0    overnight narrative accept exclusively maga cu...
1                  tax government bond fed money print
2    incredibly europe divide block alliance union ...
3    canadian security intelligence service indian ...
4    ukraine weapon european ukraine equipment desi...
5     courier deliver toy deliver blood donor hospital
6    trump road lead russia choice tie pattern inte...
7    demographic honest realise population concentr...
8                  voter afd represent vance couchfuer
9    ukraine bluesky novorossiysk ukrainian drone r...
Name: clean_text, dtype: object


## **Count Vectorization for LDA Input**

Parameters:
1. max_features=5000 – Limits vocabulary to top 5K terms.
2. max_df=0.85 – Removes words appearing in >85% of documents (too common).
3. min_df=10 – Removes words appearing in <10 documents (too rare).
4. stop_words='english' – Additional layer of English stopword removal

In [14]:
cv = CountVectorizer(
    max_features=5000,
    max_df=0.85,       # ← NEW: removes words in 85%+ of comments
    min_df=10,         # ← NEW: removes very rare words
    stop_words='english'
)

cv_matrix = cv.fit_transform(df['clean_text'])
print("Done:", cv_matrix.shape)

Done: (95008, 5000)


## **TF-IDF Vectorization for Weighted Term Representation**

Parameters:
1. max_features=5000 – Vocabulary size cap.
2. max_df=0.95 – Removes terms appearing in >95% of documents.
3. min_df=2 – Requires terms to appear in at least 2 documents.
4. stop_words='english' – Base stopword removal.

In [15]:
tfidf = TfidfVectorizer(
    max_features=5000,
    max_df=0.95,
    min_df=2,
    stop_words='english'
)

tfidf_matrix = tfidf.fit_transform(df['clean_text'])
print("Shape:", tfidf_matrix.shape)

Shape: (95008, 5000)


## **LDA Topic Model Training with Multiple Topic Configurations**

In [16]:
from sklearn.decomposition import LatentDirichletAllocation

topic_counts = [10, 15, 20]
lda_models = {}

for n in topic_counts:
    print(f"Training LDA with {n} topics...")
    lda = LatentDirichletAllocation(
        n_components=n,      # number of topics to find
        random_state=42,     # fixes randomness so results are reproducible
        max_iter=10,         # how many times to optimize (more = slower but better)
        learning_method='online'  # processes data in batches, faster for large data
    )
    lda.fit(cv_matrix)
    lda_models[n] = lda

    # Perplexity = how "confused" the model is (lower = better fit)
    print(f"  Perplexity: {lda.perplexity(cv_matrix):.2f}")

Training LDA with 10 topics...
  Perplexity: 2647.02
Training LDA with 15 topics...
  Perplexity: 2965.84
Training LDA with 20 topics...
  Perplexity: 3285.50


### **Gensim Library Installation**

Enables Gensim's LDA (faster for large corpora) and word2vec/doc2vec models

In [17]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 77.6 MB/s eta 0:00:00


### **Topic Coherence Evaluation for LDA Model Selection**

1. Convert cleaned text → list of token lists (Gensim format).
2. Create Dictionary mapping word → integer ID.
3. Extract top 10 words per topic from sklearn LDA.
4. Calculate c_v coherence (most human-interpretable metric).

In [18]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

# Step 1 — Prepare text as list of word lists
# Gensim needs: [['word1','word2'], ['word3','word4'], ...]
# NOT a full sentence string
texts = [str(text).split() for text in df['clean_text']]

# Example of what texts looks like:
# [['climate', 'change', 'real'], ['war', 'ukraine', 'bad'], ...]

# Step 2 — Build Dictionary
# Maps every unique word to a number ID
# Like a personal phonebook: climate=1, war=2, troops=3 ...
dictionary = Dictionary(texts)

# Step 3 — Function to calculate coherence
def get_coherence(lda_model, vectorizer, texts, dictionary):

    vocab = vectorizer.get_feature_names_out()  # all 5000 words
    topics = []

    for topic in lda_model.components_:
        # Get top 10 words for this topic
        top_words = [vocab[i] for i in topic.argsort()[:-11:-1]]
        topics.append(top_words)

    # CoherenceModel checks how often top words appear
    # TOGETHER in the same real comments
    # c_v is the most reliable coherence method
    cm = CoherenceModel(
        topics=topics,
        texts=texts,
        dictionary=dictionary,
        coherence='c_v'
    )
    return cm.get_coherence()

# Step 4 — Check all 3 models
print("Calculating coherence scores...\n")
scores = {}

for n in topic_counts:
    score = get_coherence(lda_models[n], cv, texts, dictionary)
    scores[n] = score
    print(f"Topics: {n}  →  Coherence Score: {score:.4f}")

# Step 5 — Pick the best
best_n = max(scores, key=scores.get)
print(f"\n✅ Best number of topics: {best_n}")
best_lda = lda_models[best_n]

Calculating coherence scores...

Topics: 10  →  Coherence Score: 0.5708
Topics: 15  →  Coherence Score: 0.5706
Topics: 20  →  Coherence Score: 0.5674

✅ Best number of topics: 10


## **LDA Model Serialization**

Saves 10-topic LDA model to lda_model.pkl

In [19]:
import pickle

with open('lda_model.pkl', 'wb') as f:
    pickle.dump(lda_models[10], f)

print("Model saved as lda_model.pkl")

Model saved as lda_model.pkl


## Topic Word Cloud Visualization

1. Extracts top 50 words per topic with their weights.
2. Generates word cloud where font size ∝ word importance.
3. Saves as high-res PNG (150 DPI).

In [20]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import os

os.makedirs('outputs', exist_ok=True)

vocab = cv.get_feature_names_out()

for i, topic in enumerate(lda_models[10].components_):
    word_weights = {vocab[j]: topic[j] for j in topic.argsort()[:-51:-1]}

    wc = WordCloud(width=800, height=400, background_color='white')
    wc.generate_from_frequencies(word_weights)

    plt.figure(figsize=(10, 5))
    plt.imshow(wc)
    plt.axis('off')
    plt.title(f'Topic {i}', fontsize=18, fontweight='bold')
    plt.savefig(f'outputs/topic_{i}_wordcloud.png', dpi=150)
    plt.close()
    print(f"Saved topic_{i}_wordcloud.png ✅")

print("\nAll word clouds done!")

Saved topic_0_wordcloud.png ✅
Saved topic_1_wordcloud.png ✅
Saved topic_2_wordcloud.png ✅
Saved topic_3_wordcloud.png ✅
Saved topic_4_wordcloud.png ✅
Saved topic_5_wordcloud.png ✅
Saved topic_6_wordcloud.png ✅
Saved topic_7_wordcloud.png ✅
Saved topic_8_wordcloud.png ✅
Saved topic_9_wordcloud.png ✅

All word clouds done!
